# ETL Notebook (Simple)

Reads a CSV from a raw path, filters out rows where `value` is null, and writes a Delta output under a processed path organized by `run_date`.

This version avoids widgets/dbutils and resolves parameters from a control table when available.

In [ ]:
from datetime import date
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()


In [ ]:
# Basic configuration + parameter resolution (widgets > control table > defaults)\n
DEFAULT_ENV = 'dev'
DEFAULT_RAW_BASE_PATH = '/Volumes/workspace/default/raw'
DEFAULT_PROCESSED_BASE_PATH = '/Volumes/workspace/default/processed'
DEFAULT_INPUT_FILENAME = 'input.csv'

def get_param(spark, env, key, explicit=None, default=None):
    if explicit is not None and str(explicit).strip() != '':
        return explicit
    try:
        df = spark.read.table('workspace.default.control_parameters')
        row = df.filter((df.env == env) & (df.key == key)).select('value').first()
        if row and row.value is not None:
            return row.value
    except Exception:
        pass
    return default

# Widgets for job/interactive parameters
try:
    dbutils.widgets.text('run_date', '')
    dbutils.widgets.text('env', '')
    dbutils.widgets.text('input_filename', '')
    dbutils.widgets.text('raw_base_path', DEFAULT_RAW_BASE_PATH)
    dbutils.widgets.text('processed_base_path', DEFAULT_PROCESSED_BASE_PATH)
except NameError:
    pass

# Read explicit values from widgets (if present)
wd_run_date = None
wd_env = None
wd_input_filename = None
wd_raw = None
wd_processed = None
try:
    wd_run_date = dbutils.widgets.get('run_date')
    wd_env = dbutils.widgets.get('env')
    wd_input_filename = dbutils.widgets.get('input_filename')
    wd_raw = dbutils.widgets.get('raw_base_path')
    wd_processed = dbutils.widgets.get('processed_base_path')
except Exception:
    pass

# Resolve env
env = (wd_env.strip() if wd_env and wd_env.strip() != '' else DEFAULT_ENV)

# Resolve run_date and paths using priority: explicit > control table > defaults
from datetime import date as _date
run_date = get_param(spark, env, 'run_date', explicit=(wd_run_date.strip() if wd_run_date and wd_run_date.strip() != '' else None), default=_date.today().strftime('%Y-%m-%d'))
raw_base_path = get_param(spark, env, 'raw_base_path', explicit=(wd_raw.strip() if wd_raw and wd_raw.strip() != '' else None), default=DEFAULT_RAW_BASE_PATH)
processed_base_path = get_param(spark, env, 'processed_base_path', explicit=(wd_processed.strip() if wd_processed and wd_processed.strip() != '' else None), default=DEFAULT_PROCESSED_BASE_PATH)
input_filename = get_param(spark, env, 'input_filename', explicit=(wd_input_filename.strip() if wd_input_filename and wd_input_filename.strip() != '' else None), default=DEFAULT_INPUT_FILENAME)

print(f'env={env}, run_date={run_date}')
print(f'raw_base_path={raw_base_path}, processed_base_path={processed_base_path}, input={input_filename}')


In [ ]:
# ETL
input_path = f"{raw_base_path}/{input_filename}"
parent_date_dir = f"{processed_base_path}/{run_date}"
output_path = f"{parent_date_dir}/Notebook"

# Ensure dbutils is available and parent folder exists (for UI visibility)
try:
    dbutils  # type: ignore[name-defined]
except NameError:
    from pyspark.dbutils import DBUtils  # type: ignore
    dbutils = DBUtils(spark)  # type: ignore

try:
    dbutils.fs.mkdirs(parent_date_dir)  # type: ignore
except Exception:
    pass

df = spark.read.option('header', True).csv(input_path)
df_filtered = df.filter(col('value').isNotNull())

print(f'Read {df.count()} rows; writing {df_filtered.count()} non-null rows to {output_path}')

df_filtered.write.format('delta').mode('overwrite').save(output_path)

# Sanity: list the parent directory to confirm presence in UI
try:
    children = [f.name for f in dbutils.fs.ls(parent_date_dir)]  # type: ignore
    print(f'Parent contents after write: {children}')
except Exception as e:
    print(f'Listing parent failed: {e}')
